In [1]:
import handcalcs.render
handcalcs.set_option("latex_block_start", "$")
handcalcs.set_option("latex_block_end", "$")
handcalcs.set_option("math_environment_start", "aligned")
handcalcs.set_option("math_environment_end", "aligned")
import math
from math import asin, sin, cos, acos, sqrt, pi, log10, degrees, radians
import pandas as pd
import forallpeople
forallpeople.environment('structural', top_level = True)
# import steelpy
# from steelpy import aisc
# import ezweld
from Pynite import FEModel3D
from Pynite.Rendering import Renderer
# import pyvista as pv

**PROJECT:**  PROJECT NAME

**PROJECT ID:**  XX-XX-XX

**DESIGNER:**  PJL

**DATE:** XX/XX/XXXX

# MODEL SETUP:

## GEOMETRY IMPORT — NODES & MEMBERS:

Node and member geometry is imported directly from the SolidWorks CSV exports (`nodes.csv`, `members.csv`) — no manual re-entry. A configurable **adapter** (`CONFIG`) maps the raw export columns onto a fixed canonical schema and sets the unit scale and coordinate snap tolerance. When a future export differs, only `CONFIG` is edited; the parse, validation, and assembly logic below is unchanged.

$\rightarrow$ *Adapter configuration*

In [2]:
# Edit ONLY this block when a new SolidWorks export format arrives.
CONFIG = {
    # raw-export column name  ->  canonical name expected by the builder
    "node_cols":   {"Node": "Node", "X": "X", "Y": "Y", "Z": "Z"},
    "member_cols": {"Member": "Member", "iNode": "iNode", "jNode": "jNode"},
    "unit_scale":  1.0,    # multiply coordinates onto model units (in -> in)
    "snap_tol":    1e-6,   # |coord| < snap_tol  ->  0.0  (removes CAD float noise)
}

$\rightarrow$ *Parser*

In [3]:
def load_nodes(path, cfg):
    """Read node CSV -> canonical DataFrame [Node, X, Y, Z], cleaned."""
    df = pd.read_csv(path).rename(columns={v: k for k, v in cfg["node_cols"].items()})
    df = df[["Node", "X", "Y", "Z"]].copy()
    for ax in ("X", "Y", "Z"):
        df[ax] = pd.to_numeric(df[ax], errors="coerce") * cfg["unit_scale"]
        df.loc[df[ax].abs() < cfg["snap_tol"], ax] = 0.0   # snap float noise
    return df

def load_members(path, cfg):
    """Read member CSV -> canonical DataFrame [Member, iNode, jNode]."""
    df = pd.read_csv(path).rename(columns={v: k for k, v in cfg["member_cols"].items()})
    return df[["Member", "iNode", "jNode"]].copy()

$\rightarrow$ *Referential-integrity validation*

Fails loudly on the failure modes a varying CAD export can introduce: duplicate names, missing/non-numeric coordinates, members referencing a non-existent node, and zero-length members. A malformed export becomes an explicit error rather than a silently wrong model.

In [4]:
def validate(nodes, members):
    errs = []
    if nodes["Node"].duplicated().any():
        errs.append(f"Duplicate node names: {list(nodes['Node'][nodes['Node'].duplicated()])}")
    if members["Member"].duplicated().any():
        errs.append(f"Duplicate member names: {list(members['Member'][members['Member'].duplicated()])}")
    if nodes[["X", "Y", "Z"]].isna().any().any():
        errs.append("Non-numeric / missing coordinates present.")
    known = set(nodes["Node"])
    for _, r in members.iterrows():
        for end in ("iNode", "jNode"):
            if r[end] not in known:
                errs.append(f"{r['Member']}: {end} '{r[end]}' not found in node table.")
        if r["iNode"] == r["jNode"]:
            errs.append(f"{r['Member']}: zero-length (iNode == jNode).")
    if errs:
        raise ValueError("Validation failed:\n  - " + "\n  - ".join(errs))
    print(f"Validation passed: {len(nodes)} nodes, {len(members)} members.")
    return True

## MODEL ASSEMBLY:

PyNite requires a material **and** a section to exist before any member can be added, so a single **placeholder** material and section are registered purely to let the members instantiate for this geometry pass. They carry **no structural meaning** and are replaced when real properties (via `steelpy` or manual input) are introduced in a later step.

In [5]:
def build_model(nodes, members):
    model = FEModel3D()

    # --- PLACEHOLDER material & section (geometry pass only; no meaning) ---
    model.add_material("PLACEHOLDER", E=1.0, G=1.0, nu=0.3, rho=1.0)
    model.add_section("PLACEHOLDER", A=1.0, Iy=1.0, Iz=1.0, J=1.0)

    for _, r in nodes.iterrows():
        model.add_node(r["Node"], r["X"], r["Y"], r["Z"])
    for _, r in members.iterrows():
        model.add_member(r["Member"], r["iNode"], r["jNode"],
                         "PLACEHOLDER", "PLACEHOLDER")
    return model

$\rightarrow$ *Build*

In [6]:
nodes   = load_nodes("nodes.csv", CONFIG)
members = load_members("members.csv", CONFIG)
validate(nodes, members)
model = build_model(nodes, members)

print(f"Model assembled: {len(model.nodes)} nodes, {len(model.members)} members.")

Validation passed: 30 nodes, 34 members.
Model assembled: 30 nodes, 34 members.


## WIREFRAME:

Rendered with PyNite's built-in **pyvista** renderer. `render_model()` opens an interactive window; run this cell from the Jupyter kernel (not the Quarto→PDF render). `annotation_size` is in model units (inches) and is tuned to the ~600 in model extent — adjust to taste. To capture a static image for the deliverable, use the commented `screenshot()` call instead.

In [9]:
import pyvista as pv
pv.global_theme.notebook = False   # render in a standalone pop-up window, not in the cell

rndr = Renderer(model)
rndr.annotation_size = 10
rndr.render_nodes = True
rndr.show_labels   = True
rndr.render_loads  = False

# Clickable orientation gizmo — click an axis to snap to X / Y / Z / iso views
rndr.post_update_callbacks.append(lambda p: p.add_camera_orientation_widget())

rndr.render_model()   # opens an interactive desktop window